# Candidatos, visibilidad y curvas de luz (sin LSST)

Carga candidatos desde TOMs configurados y catálogos de eventos de surveys locales. Calcula visibilidad para El Leoncito, descarta targets que no pasan el corte, guarda la tabla por noche y muestra las curvas de luz de los targets visibles.

Edita `CONFIG_PATH`, fechas y cortes. Las consultas TOM y OGLE pueden necesitar red y credenciales; las tablas locales requieren inventarios configurados. Este flujo no importa ni consulta LSST/Rubin.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

from target_selection.config import load_config
from target_selection.cache_paths import CACHE_ROOT, OUTPUT_ROOT
from target_selection.planning import (
    collect_photometry, filter_targets_with_valid_coordinates,
    load_candidates, load_ogle_ews_candidates, merge_candidate_targets,
    evaluate_visibility, visibility_plots,
)
from target_selection.planning_cli import _load_local_surveys
from target_selection.report import create_lightcurve_plots
from target_selection.run_paths import safe_name
from target_selection.tasks.surveys import load_observed_survey_targets

CONFIG_PATH = Path('../configs/local_hsh.toml')
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path('configs/local_hsh.toml')
config = load_config(CONFIG_PATH)
START_DATE = (pd.Timestamp.now(tz='UTC') + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
END_DATE = (pd.Timestamp(START_DATE) + pd.Timedelta(days=2)).strftime('%Y-%m-%d')
OBSERVATORY = config.observatory
MIN_ALTITUDE_DEG = config.selection.minimum_altitude_deg
MIN_VISIBLE_MINUTES = config.selection.minimum_observable_minutes
OBSERVING_WINDOWS = config.selection.observing_windows
TOM_PROVIDERS = tuple(config.target_providers)
INCLUDE_OGLE_EWS = False  # True agrega el catálogo público OGLE EWS.
OUTPUT_DIR = OUTPUT_ROOT / 'runs' / 'tom_survey_visibility_lightcurves'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Configuración: {CONFIG_PATH.resolve()}')
print(f'TOMs: {TOM_PROVIDERS}; fechas: {START_DATE} a {END_DATE}')
print(f'Corte: altitud ≥ {MIN_ALTITUDE_DEG}°, visibilidad ≥ {MIN_VISIBLE_MINUTES} min')


## 1. Candidatos por fuente

Cada TOM se consulta por separado para conservar su procedencia. Los surveys locales aportan targets observados con coordenadas del catálogo de eventos, no de la posición del telescopio.


In [ ]:
provider_settings = {
    name: dict(config.provider_specs[name].options)
    for name in TOM_PROVIDERS if name in config.provider_specs
}
candidate_frames = []
candidates_by_source = {}

for provider in TOM_PROVIDERS:
    frame = load_candidates(
        providers=(provider,), start_date=START_DATE, end_date=END_DATE,
        observatory=OBSERVATORY, minimum_altitude_deg=MIN_ALTITUDE_DEG,
        minimum_observable_minutes=MIN_VISIBLE_MINUTES,
        observing_windows=OBSERVING_WINDOWS, provider_settings=provider_settings,
        cache_dir=CACHE_ROOT, verbose=True,
    )
    candidates_by_source[provider] = frame
    candidate_frames.append(frame)
    print(f'{provider}: {len(frame)} candidatos')
    display(frame.head(10))

def configured_path(spec, key, default):
    path = Path(spec.options.get(key, default))
    if path.is_absolute():
        return path
    base_dir = Path(spec.options.get('config_dir', Path.cwd()))
    return (base_dir / path).resolve()

for name in config.followup_surveys:
    spec = config.followup_specs[name]
    provider = str(spec.options.get('provider_name', name)).upper()
    source_args = {'providers': (provider,), 'verbose': True}
    if provider == 'HSH':
        source_args.update(
            hsh_inventory_path=configured_path(spec, 'inventory_path', 'data/hsh_data/image_collection_astro.csv'),
            hsh_target_catalog_path=configured_path(spec, 'target_catalog_path', 'data/hsh_data/hsh_objects.csv'),
        )
    elif provider == 'JS':
        source_args['js_inventory_path'] = configured_path(spec, 'inventory_path', 'data/js_data/js_inventory.csv')

    frame = load_observed_survey_targets(**source_args)
    candidates_by_source[name] = frame
    candidate_frames.append(frame)
    print(f'{name} ({provider}): {len(frame)} candidatos')
    display(frame.head(10))

if INCLUDE_OGLE_EWS:
    ogle_candidates = load_ogle_ews_candidates(cache_dir=CACHE_ROOT, verbose=True)
    candidates_by_source['ogle_ews'] = ogle_candidates
    candidate_frames.append(ogle_candidates)
    print(f'OGLE EWS: {len(ogle_candidates)} candidatos')
    display(ogle_candidates.head(10))

candidates = filter_targets_with_valid_coordinates(merge_candidate_targets(*candidate_frames))
print(f'Total único con coordenadas válidas: {len(candidates)}')
display(candidates.head(20))
if candidates.empty:
    raise RuntimeError('No hay candidatos con coordenadas. Revisa TOMs, inventarios y credenciales.')


## 2. Visibilidad y corte

`selected_for_visibility` aplica los criterios configurados de altitud y minutos observables. Se conserva una fila por target y noche, un resumen por target y la selección con al menos una noche visible.


In [ ]:
visibility_dir = OUTPUT_DIR / 'visibility'
visibility_by_night, visibility_summary, _ = evaluate_visibility(
    candidates, visibility_dir, start_date=START_DATE, end_date=END_DATE,
    observatory=OBSERVATORY, minimum_altitude_deg=MIN_ALTITUDE_DEG,
    minimum_observable_minutes=MIN_VISIBLE_MINUTES,
    observing_windows=OBSERVING_WINDOWS, verbose=True,
)
visible_ids = set(visibility_summary.loc[
    visibility_summary['n_visible_nights'].gt(0), 'target_id'
].astype(str))
visible_targets = candidates.loc[
    candidates['target_id'].astype(str).isin(visible_ids)
].copy()

print(f'{len(visible_targets)} de {len(candidates)} targets pasan el corte')
display(visibility_summary.sort_values(
    ['n_visible_nights', 'total_observable_minutes'], ascending=False
).reset_index(drop=True))
print(f'Tabla por noche: {visibility_dir / "visibility_by_night.csv"}')
print(f'Tabla resumen: {visibility_dir / "visibility_summary.csv"}')
display(visibility_by_night.head(20))


In [ ]:
visibility_images = visibility_plots(
    candidates, visibility_dir / 'plots', start_date=START_DATE, end_date=END_DATE,
    observatory=OBSERVATORY, minimum_altitude_deg=MIN_ALTITUDE_DEG,
    minimum_observable_minutes=MIN_VISIBLE_MINUTES,
    observing_windows=OBSERVING_WINDOWS, selection=visibility_by_night,
    overwrite=True, verbose=True,
)
for path in visibility_images:
    display(Markdown(f'**{path.stem}**'))
    display(Image(filename=str(path)))


## 3. Fotometría y curvas de luz de los visibles

Las consultas de fotometría se hacen después del corte. Se combinan puntos de los TOMs con fotometría de inventarios locales configurados. Los PNG se guardan y se muestran aquí, target por target.


In [ ]:
if not visible_targets.empty:
    tom_photometry = collect_photometry(
        visible_targets, providers=TOM_PROVIDERS, query_all_tom_providers=True,
        include_associated_surveys=True, cache_dir=CACHE_ROOT, verbose=True,
    )
    survey_epochs, local_photometry = _load_local_surveys(config, visible_targets, verbose=True)
    photometry_parts = [frame for frame in (tom_photometry, local_photometry)
                        if frame is not None and not frame.empty]
    photometry = pd.concat(photometry_parts, ignore_index=True, sort=False) if photometry_parts else pd.DataFrame()
else:
    tom_photometry = pd.DataFrame()
    survey_epochs = pd.DataFrame()
    photometry = pd.DataFrame()

print(f'Fotometría: {len(photometry)} puntos; épocas locales: {len(survey_epochs)}')
display(photometry.head(20))


In [ ]:
provider_layers = {
    'mop': 'mop_photometry', 'opm': 'opm_photometry',
    'bhtom': 'bhtom_photometry',
}
layers = [provider_layers[name] for name in TOM_PROVIDERS if name in provider_layers]
layers.extend(
    'hsh' if config.followup_specs[name].options.get('provider_name', name).upper() == 'HSH' else 'js'
    for name in config.followup_surveys
)
if any(str(value).upper() == 'OGLE' for value in photometry.get('survey', pd.Series(dtype=str))):
    layers.append('ogle_photometry')
layers = tuple(dict.fromkeys(layers))

curve_dir = OUTPUT_DIR / 'lightcurves'
curve_paths = create_lightcurve_plots(
    visible_targets, output_dir=curve_dir,
    observatory_epochs=survey_epochs, observatory_photometry=photometry,
    layers=layers, observation_date=START_DATE, overwrite=True, verbose=True,
) if not visible_targets.empty else []

for _, target in visible_targets.iterrows():
    target_id = str(target['target_id'])
    matches = [path for path in curve_paths if path.stem.casefold() == safe_name(target_id).casefold()]
    if matches:
        display(Markdown(f'### {target_id}'))
        display(Image(filename=str(matches[0])))
    else:
        print(f'Sin curva generada para {target_id}')
print(f'Curvas guardadas en {curve_dir}')
